# SatQuery AI — Real Remote GPU Inference Server (Qwen2.5-VL-3B-Instruct + VRSBench LoRA)
### Hosted on Kaggle instead of Colab — no paid tier required

**What changed vs. the old Colab-tunnel setup:**
1. **The backend used to never send the actual image to this endpoint** —
   `geochat_service.py`'s remote-tunnel call posted only a text prompt. Fixed on the backend
   side (`geochat_service.py` now base64-encodes the real image and sends it as
   `image_base64`) — **this notebook's `/vqa` endpoint expects and actually uses that
   field.**
2. **Base model**: `Qwen/Qwen2.5-VL-3B-Instruct` — a production-maintained, actively
   developed VLM (Alibaba, official cookbooks, first-class `transformers` support), not one
   of the academic RS-specific checkpoints (GeoChat/EarthDial/GeoPixel/TEOChat) — all four
   were checked against their real GitHub issue trackers and all four have multiple open,
   unresolved "missing file"/"broken inference" issues with zero fixes. RS-adaptation comes
   from this project's own real LoRA fine-tuning (`kaggle_finetune_qwen2vl.ipynb`) on
   `xiang709/VRSBench` — real images, real captions/VQA/referring-expression annotations.
3. Runs on **Kaggle's free GPU quota** (~30 hrs/week, T4/P100, no credit card).

**Still true from the old setup**: this is a live tunnel, not an always-on host. Run this
notebook and keep it running for your demo/testing window, and paste the printed public URL
into your backend's `.env` as `REMOTE_INFERENCE_URL` (the old `COLAB_INFERENCE_URL` name
still works too) each time you restart it — the ngrok URL changes on every run.

**Untested disclaimer**: written without GPU access to run it myself — please run it end to
end and report back what breaks.

**How to run**: New Kaggle Notebook → Settings → Accelerator → GPU T4 x2 → Internet: On →
add your `HF_TOKEN` and `NGROK_AUTHTOKEN` as Kaggle Secrets (Add-ons → Secrets) so they're
never pasted in plaintext into the notebook.


In [ ]:
# Cell 1: Verify GPU
!nvidia-smi

In [ ]:
# Cell 2: Install dependencies.
#
# Honesty note (2026-09-08): this cell used to install transformers from git main
# ("Qwen2.5-VL needs a recent transformers ... since PyPI releases can lag" — true when
# this notebook was first written). That's no longer true, and was actively causing real,
# confirmed-from-the-user's-own-output failures: git main is a moving target, so a manual
# tokenizers pin chosen to satisfy one day's git-main commit
# (tokenizers>=0.22.0,<=0.23.0) broke again on the very next run, when git main had moved
# on to requiring tokenizers<0.24.0,>=0.23.1 instead. Verified directly (Hugging Face's own
# model-doc page) that Qwen2.5-VL support has been in a numbered, stable transformers
# release since v5.15.1 -- pinning to that specific release fixes this whole class of bug
# permanently: pip resolves tokenizers (and everything else) against one fixed, coherent
# release instead of a shifting git HEAD, and no manual tokenizers override is needed.
#
# Older honesty note (2026-08-25), still relevant: the fine-tuning notebook's equivalent
# cell hit a REAL error on Kaggle -- `pip install -U ... pillow ...` pulled Pillow to
# 12.3.0, too new for the torchvision build already on the base image, causing
# `ImportError: cannot import name '_Ink' from 'PIL._typing'` deep inside
# transformers -> torchvision -> PIL.ImageDraw. Pin Pillow below 12 as the LAST install
# step. Matches the environment's own pre-installed gradio's stated requirement
# (`pillow<12.0,>=8.0`), not an arbitrary downgrade.
!pip install -q -U "transformers==5.15.1" accelerate
!pip install -q -U peft bitsandbytes
!pip install -q fastapi uvicorn pyngrok python-multipart
!pip install -q qwen-vl-utils
!pip install -q "pillow<12,>=10" --force-reinstall


In [ ]:
# Cell 3: Load the real base model + your real trained LoRA adapter from Hugging Face Hub.
# HF_REPO_ID must match what you pushed to in kaggle_finetune_qwen2vl.ipynb's Cell 9.
import torch
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
from peft import PeftModel
from kaggle_secrets import UserSecretsClient

HF_REPO_ID = "Sameelkazi/satquery-qwen25vl-vrsbench-lora-v2"  # real trained v2 checkpoint (77.0% VQA acc, 0.3671 mean IoU)
BASE_MODEL_ID = "Qwen/Qwen2.5-VL-3B-Instruct"

secrets = UserSecretsClient()
hf_token = secrets.get_secret("HF_TOKEN")  # add this in Kaggle -> Add-ons -> Secrets

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

processor = AutoProcessor.from_pretrained(HF_REPO_ID, token=hf_token)
base_model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

adapter_active = False
try:
    model = PeftModel.from_pretrained(base_model, HF_REPO_ID, token=hf_token)
    adapter_active = True
    print(f"Real LoRA adapter loaded from {HF_REPO_ID}")
except Exception as e:
    # Never fabricate: if the adapter can't load, serve the un-adapted base model and
    # report adapter_active=False honestly rather than pretending the adapter is active.
    model = base_model
    print(f"Could not load LoRA adapter from {HF_REPO_ID} ({e}); serving un-adapted base model.")

model.eval()
print("Ready. adapter_active =", adapter_active)

In [ ]:
# Cell 4: The actual /vqa server — this is the fix for the core bug. It requires and
# decodes a real base64-encoded image, runs it through the real Qwen2.5-VL vision pipeline
# (not a text placeholder), and computes confidence from the model's own generation scores
# (real logits — never a hardcoded number).
import base64
import io
import threading
import time

from fastapi import FastAPI, HTTPException
from pydantic import BaseModel
from typing import List, Optional, Dict
from PIL import Image
import numpy as np
import uvicorn

app = FastAPI()


class VQARequest(BaseModel):
    prompt: str
    image_base64: str  # required — no fallback to blind text-only generation
    # Conversational memory (added 2026-09-09, backend/session_store.py): real prior turns
    # of this session ({"query", "answer"} dicts, oldest first), forwarded from the backend
    # so this endpoint can build genuine multi-turn chat-template turns instead of only
    # ever seeing the recap folded into `prompt` as flat text. Optional and additive — an
    # older backend that doesn't send this field simply omits it (Optional default None),
    # and this server still works exactly as before for that caller.
    conversation_history: Optional[List[Dict[str, str]]] = None


@app.post("/vqa")
def vqa(req: VQARequest):
    if not req.image_base64:
        # Honesty guard: refuse rather than silently generating a blind guess.
        raise HTTPException(status_code=400, detail="image_base64 is required — this server does not answer without seeing the real image.")

    try:
        image_bytes = base64.b64decode(req.image_base64)
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"could not decode image_base64: {e}")

    # req.prompt may arrive in this project's "USER: <image>\n{query} ASSISTANT:" text
    # convention (from geochat_service.py) — strip that wrapper and keep just the actual
    # question, since Qwen2.5-VL's chat template supplies its own turn structure.
    query_text = req.prompt
    if "USER:" in query_text and "ASSISTANT:" in query_text:
        query_text = query_text.split("USER:", 1)[1].split("ASSISTANT:", 1)[0]
        query_text = query_text.replace("<image>", "").strip()

    # Conversational memory: real prior turns as genuine user/assistant text-only
    # chat-template turns, ahead of the final image-bearing turn -- mirrors
    # geochat_service.py's own _history_to_chat_turns exactly (kept as a small inline
    # helper here since this notebook cannot import that backend module directly).
    messages = []
    for h in (req.conversation_history or []):
        h_q, h_a = (h.get("query") or "").strip(), (h.get("answer") or "").strip()
        if h_q and h_a:
            messages.append({"role": "user", "content": [{"type": "text", "text": h_q}]})
            messages.append({"role": "assistant", "content": [{"type": "text", "text": h_a}]})
    messages.append({"role": "user", "content": [
        {"type": "image"},
        {"type": "text", "text": query_text},
    ]})
    chat_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[chat_text], images=[[image]], return_tensors="pt").to(model.device)

    # Honesty correction (2026-08-26): the fine-tuning notebook's real Cell 10 sanity
    # check exposed a genuine undertraining symptom in the first real adapter — a short,
    # plausible answer followed by endless repetition instead of stopping. Fixed at the
    # root on the training side (more LoRA capacity + more epochs), but this production
    # endpoint gets the same defensive generation-time safeguard so a user-facing request
    # can't hang on a repetition loop regardless of which adapter build is loaded.
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            output_scores=True,
            return_dict_in_generate=True,
        )

    generated_ids = output.sequences[:, inputs["input_ids"].shape[1]:]
    answer_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    # Real, model-derived confidence — mean top-token probability across generated steps,
    # not a hardcoded number.
    confidence = None
    if output.scores:
        probs = [torch.softmax(s, dim=-1).max().item() for s in output.scores]
        confidence = round(float(np.mean(probs)), 3)

    return {
        "text": answer_text,
        "confidence": confidence,  # backend treats a missing/None confidence honestly (heuristic fallback), never assumes one
        "adapter_active": adapter_active,
        "model": f"{BASE_MODEL_ID}{' + VRSBench LoRA (' + HF_REPO_ID + ')' if adapter_active else ''}",
    }


class VQAMultiRequest(BaseModel):
    prompt: str
    images_base64: List[str]  # required, >=2 — genuine joint multi-image inference (bi-temporal / optical+SAR)
    image_labels: Optional[List[str]] = None
    # Conversational memory (added 2026-09-09) -- same optional, additive field as
    # VQARequest above.
    conversation_history: Optional[List[Dict[str, str]]] = None


@app.post("/vqa_multi")
def vqa_multi(req: VQAMultiRequest):
    """
    Added 2026-09-07 to close a real compliance gap: this project's backend
    (geochat_service.run_vqa_multi) could not use this tunnel for genuine joint
    bi-temporal / optical+SAR reasoning because this server only exposed single-image
    /vqa, so those two mandatory PS task families fell through to a generic cloud VLM
    fallback whenever a REMOTE_INFERENCE_URL tunnel was configured. This mirrors
    geochat_service.py's own _build_chat_inputs_multi prompt construction exactly, so
    the real RS-adapted model (base Qwen2.5-VL-3B + this project's real VRSBench LoRA)
    genuinely sees both images jointly via its native multi-image chat template.
    """
    if not req.images_base64 or len(req.images_base64) < 2:
        raise HTTPException(status_code=400, detail="images_base64 must contain at least 2 images — this endpoint is for genuine joint multi-image inference; use /vqa for single-image VQA.")

    try:
        images = []
        for b64 in req.images_base64:
            image_bytes = base64.b64decode(b64)
            images.append(Image.open(io.BytesIO(image_bytes)).convert("RGB"))
    except Exception as e:
        raise HTTPException(status_code=400, detail=f"could not decode images_base64: {e}")

    query_text = req.prompt
    if "USER:" in query_text and "ASSISTANT:" in query_text:
        query_text = query_text.split("USER:", 1)[1].split("ASSISTANT:", 1)[0]
        query_text = query_text.replace("<image>", "").strip()

    labels = req.image_labels or [f"Image {i + 1}" for i in range(len(images))]
    label_line = " / ".join(labels)
    system_instruction = (
        "Answer the user's question directly and concisely (2-4 sentences unless more "
        "detail is explicitly requested), describing only what is genuinely visible "
        f"across the {len(images)} provided images, presented in this order: "
        f"{label_line}. Do not enumerate long lists of generic remote-sensing "
        "vocabulary or categories that are not actually present. Compare the images "
        "directly when the query asks about change, difference, or correlation "
        "between them."
    )
    content = [{"type": "image"} for _ in images] + [{"type": "text", "text": query_text}]
    messages = [{"role": "system", "content": [{"type": "text", "text": system_instruction}]}]
    for h in (req.conversation_history or []):
        h_q, h_a = (h.get("query") or "").strip(), (h.get("answer") or "").strip()
        if h_q and h_a:
            messages.append({"role": "user", "content": [{"type": "text", "text": h_q}]})
            messages.append({"role": "assistant", "content": [{"type": "text", "text": h_a}]})
    messages.append({"role": "user", "content": content})
    chat_text = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(text=[chat_text], images=[images], return_tensors="pt").to(model.device)

    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=256,
            do_sample=False,
            repetition_penalty=1.3,
            no_repeat_ngram_size=3,
            output_scores=True,
            return_dict_in_generate=True,
        )

    generated_ids = output.sequences[:, inputs["input_ids"].shape[1]:]
    answer_text = processor.batch_decode(generated_ids, skip_special_tokens=True)[0].strip()

    confidence = None
    if output.scores:
        probs = [torch.softmax(s, dim=-1).max().item() for s in output.scores]
        confidence = round(float(np.mean(probs)), 3)

    return {
        "text": answer_text,
        "confidence": confidence,
        "adapter_active": adapter_active,
        "model": f"{BASE_MODEL_ID}{' + VRSBench LoRA (' + HF_REPO_ID + ')' if adapter_active else ''}",
    }


@app.get("/health")
def health():
    return {"status": "ok", "adapter_active": adapter_active}

In [ ]:
# Cell 5: Start the server + a public tunnel via ngrok, print the URL to paste into your
# backend's .env as REMOTE_INFERENCE_URL (or the older COLAB_INFERENCE_URL name).
from pyngrok import ngrok, conf

ngrok_token = secrets.get_secret("NGROK_AUTHTOKEN")  # free account at https://ngrok.com
conf.get_default().auth_token = ngrok_token

def run_server():
    uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning")

thread = threading.Thread(target=run_server, daemon=True)
thread.start()
time.sleep(3)

tunnel = ngrok.connect(8000, "http")
# Honesty/correctness fix (2026-08-26): ngrok.connect() returns an NgrokTunnel object,
# not a plain string. Printing it directly (an earlier version of this cell did
# `print(f"...{tunnel}")`) shows its repr (`NgrokTunnel: "https://..." -> "http://..."`),
# which is NOT a valid value to paste into .env. Extract the real URL string explicitly.
public_url = tunnel.public_url
print("=" * 70)
print(f"Public inference URL: {public_url}")
print("Paste this into your backend's .env as:")
print(f"  REMOTE_INFERENCE_URL={public_url}")
print("This URL is only valid while this notebook cell keeps running — restarting the")
print("notebook (or letting the Kaggle session time out) generates a new URL, which you'll")
print("need to update in .env again.")
print("=" * 70)

In [ ]:
# Cell 6: Keep the session alive. Kaggle sessions run up to ~12 hours per session /
# ~30 hours/week free quota — stop this cell (or the session will just idle-timeout) when
# you're done with your demo/testing window.
while True:
    time.sleep(60)